# GPU Parallelism & CUDA Basics (Numba)

**No local NVIDIA GPU?** Upload this notebook to Google Colab (Runtime → Change
runtime type → GPU) — every cell guards on GPU availability and tells you what it
*would* show.

## 1. Confirm GPU availability

In [ ]:
import numpy as np

try:
    from numba import cuda
    GPU = cuda.is_available()
except ImportError:
    cuda, GPU = None, False

print("GPU available:", GPU)
if GPU:
    print(cuda.get_current_device().name.decode())

## 2. A first kernel: elementwise vector add, one thread per element

In [ ]:
if GPU:
    from numba import cuda

    @cuda.jit
    def vector_add(a, b, out):
        i = cuda.grid(1)              # this thread's global index
        if i < out.size:              # grid may be slightly larger than the data
            out[i] = a[i] + b[i]

    n = 1_000_000
    a = np.random.rand(n).astype(np.float32)
    b = np.random.rand(n).astype(np.float32)
    out = np.zeros_like(a)

    threads_per_block = 256
    blocks = (n + threads_per_block - 1) // threads_per_block
    vector_add[blocks, threads_per_block](a, b, out)

    assert np.allclose(out, a + b)
    print(f"kernel correct — launched {blocks} blocks x {threads_per_block} threads")
else:
    print("[no GPU] The kernel maps thread i -> element i; the guard `i < out.size`")
    print("handles the last partial block. Run on Colab to execute.")

## 3. Benchmark vs. NumPy at increasing sizes

In [ ]:
import time

def bench_cpu(n, reps=5):
    a, b = np.random.rand(n).astype(np.float32), np.random.rand(n).astype(np.float32)
    t0 = time.perf_counter()
    for _ in range(reps):
        a + b
    return (time.perf_counter() - t0) / reps

sizes = [10**4, 10**5, 10**6, 10**7, 10**8]
cpu_times = {n: bench_cpu(n) for n in sizes if n <= 10**7 or GPU}

if GPU:
    def bench_gpu(n, reps=5):
        a = cuda.to_device(np.random.rand(n).astype(np.float32))
        b = cuda.to_device(np.random.rand(n).astype(np.float32))
        out = cuda.device_array(n, dtype=np.float32)
        tpb = 256
        blocks = (n + tpb - 1) // tpb
        vector_add[blocks, tpb](a, b, out); cuda.synchronize()  # warm-up + JIT
        t0 = time.perf_counter()
        for _ in range(reps):
            vector_add[blocks, tpb](a, b, out)
        cuda.synchronize()
        return (time.perf_counter() - t0) / reps

    gpu_times = {n: bench_gpu(n) for n in sizes}
    for n in sizes:
        if n in cpu_times:
            print(f"n=1e{int(np.log10(n))}: CPU {cpu_times[n]*1e3:8.3f} ms | "
                  f"GPU {gpu_times[n]*1e3:8.3f} ms | speedup x{cpu_times[n]/gpu_times[n]:.1f}")
else:
    for n, t in cpu_times.items():
        print(f"n=1e{int(np.log10(n))}: CPU {t*1e3:8.3f} ms")
    print("[no GPU] On Colab expect the GPU to lose at 1e4-1e5 and win 10-100x at 1e8.")

## 4. The transfer tax: time host→device copies separately from compute

In [ ]:
if GPU:
    n = 10**7
    host = np.random.rand(n).astype(np.float32)

    t0 = time.perf_counter(); device = cuda.to_device(host); cuda.synchronize()
    t_transfer = time.perf_counter() - t0

    out = cuda.device_array(n, dtype=np.float32)
    tpb = 256; blocks = (n + tpb - 1) // tpb
    vector_add[blocks, tpb](device, device, out); cuda.synchronize()
    t0 = time.perf_counter()
    vector_add[blocks, tpb](device, device, out); cuda.synchronize()
    t_compute = time.perf_counter() - t0

    print(f"transfer 40MB host->device: {t_transfer*1e3:.2f} ms")
    print(f"one add kernel on-device:   {t_compute*1e3:.2f} ms")
    print(f"transfer = {t_transfer/t_compute:.0f}x the kernel — keep data resident!")
else:
    print("[no GPU] Typical result: one PCIe copy costs tens of kernel launches.")
    print("Moral: chain many kernels per transfer, only copy small results back.")

## 5. Mini-project: find the crossover point

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(8, 4.5))
xs = sorted(cpu_times)
plt.loglog(xs, [cpu_times[n] for n in xs], "o-", label="CPU (NumPy)")
if GPU:
    plt.loglog(sizes, [gpu_times[n] for n in sizes], "s-", label="GPU (Numba)")
    plt.title("Find where the curves cross — that's your crossover size")
else:
    plt.title("CPU baseline — add the GPU curve on Colab")
plt.xlabel("array size"); plt.ylabel("seconds per op"); plt.legend(); plt.grid(alpha=0.3)
plt.show()
# Extend: replace vector add with a distance-matrix kernel (each thread computes
# one pairwise distance) and re-find the crossover — it moves left as the
# arithmetic per element grows.